# Rogers Pass Snow Profiles Data Exploration

This notebook profiles the raw March 2025 Rogers Pass campaign package for FRDR README drafting.


In [1]:
from __future__ import annotations

import csv
from collections import Counter
from pathlib import Path

import pandas as pd
from snowmicropyn import Profile

BASE = Path.cwd()
if not (BASE / "datasets").exists():
    BASE = BASE.parent
ROOT = BASE / "datasets/rogers_pass_snow_profiles/raw_data/Rogers Pass March 2024-2025"
ROOT


WindowsPath('C:/Users/beav3503/dev/grimp_frdr_helper/datasets/rogers_pass_snow_profiles/raw_data/Rogers Pass March 2024-2025')

In [2]:
inventory = Counter(p.suffix for p in ROOT.rglob("*") if p.is_file())
inventory_rows = [{"extension": ext or "<none>", "count": count} for ext, count in sorted(inventory.items())]
pd.DataFrame(inventory_rows, columns=["extension", "count"]).sort_values(["count", "extension"], ascending=[False, True]).reset_index(drop=True)


,extension,count
0,.csv,322
1,.txt,166
2,.pnt,79
3,.HEIC,22
4,.pdf,17
5,.xlsx,13
6,.docx,11
7,.TXT,6
8,.heic,3
9,.zip,3


In [3]:
def parse_snowscope(path: Path) -> tuple[dict[str, str], pd.DataFrame]:
    lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    meta: dict[str, str] = {}
    start = None
    for idx, line in enumerate(lines):
        parts = list(csv.reader([line]))[0]
        if parts and parts[0] == "depth (mm)":
            start = idx + 1
            break
        if len(parts) >= 2:
            meta[parts[0].strip()] = ",".join(parts[1:]).strip()
    rows = []
    if start is not None:
        for line in lines[start:]:
            parts = list(csv.reader([line]))[0]
            if len(parts) != 3:
                continue
            try:
                depth = float(parts[0])
                hardness = float(parts[1])
            except ValueError:
                continue
            optical = parts[2].strip()
            rows.append(
                {
                    "depth_mm": depth,
                    "hardness_kpa": hardness,
                    "optical_reflectance_avg": None if optical.lower() in {"", "null"} else float(optical),
                }
            )
    return meta, pd.DataFrame(rows)


snowscope_files = sorted(
    p for p in ROOT.rglob("*.csv")
    if p.name not in {"Radar Fidelity.csv", "jimbaycorner_01032025.csv", "ROUND HILL.csv"}
)
meta, profile = parse_snowscope(snowscope_files[0])
meta, profile.head()


({'name': 'null',
  'elevation (m)': '1865.9941517824384',
  'aspect': 'null',
  'air temperature (c)': 'null',
  'slope angle (deg)': 'null',
  'profilePrivacy': 'public',
  'totalSnowDepth (cm)': 'null',
  'freeText': 'null',
  'collectionTime': 'Mar 1 2025 13:36 heure normale de l’Est nord-américain',
  'collectionTime (Unix Time)': '1740854211',
  'creator name': 'Francis Gauthier',
  'org name': 'null',
  'Location': '51.2365204,-117.7008331',
  'testNum': '43',
  'serialNum': '00328',
  'profileDepth (mm)': '1781',
  'batteryCapacity': '0',
  'errorCode': '0',
  'temperature': '9',
  'FW_version': '2.4.1',
  'PCB_version': 'v2.7'},
    depth_mm  hardness_kpa  optical_reflectance_avg
 0       1.0          3.28                   2073.0
 1       2.0          3.28                   2073.0
 2       3.0          2.93                   2073.0
 3       4.0          3.13                   2112.0
 4       5.0          4.17                   2112.0)

In [4]:
snowscope_summary = []
for path in snowscope_files:
    meta, profile = parse_snowscope(path)
    snowscope_summary.append(
        {
            "path": str(path.relative_to(ROOT)),
            "serial": meta.get("serialNum"),
            "creator": meta.get("creator name"),
            "profile_depth_mm": pd.to_numeric(meta.get("profileDepth (mm)"), errors="coerce"),
            "rows": len(profile),
            "has_optical": bool(profile["optical_reflectance_avg"].notna().any()) if not profile.empty else False,
            "latlon": meta.get("Location"),
        }
    )
snowscope_df = pd.DataFrame(snowscope_summary)
snowscope_df.groupby(["serial", "has_optical"]).size().reset_index(name="file_count")


,serial,has_optical,file_count
0,00304,True,71
1,00322,False,138
2,00328,True,110


In [5]:
def radar_block_counts(path: Path) -> dict[str, object]:
    lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    block_headers = sum(1 for line in lines if line.startswith("X (m),"))
    row_count = sum(1 for line in lines if line.count(",") == 4 and line[:1].isdigit())
    return {
        "path": str(path.relative_to(ROOT)),
        "block_headers": block_headers,
        "row_count": row_count,
    }


radar_files = sorted(p for p in ROOT.rglob("*.txt") if "radar_" in str(p).lower())
pd.DataFrame(radar_block_counts(path) for path in radar_files).describe(include="all")


,path,block_headers,row_count
count,164,164.0,164.0
unique,164,NaN,NaN
top,Jour 1 - Fidelity\Spatial_Jimbaycorner\radar_k...,NaN,NaN
freq,1,NaN,NaN
mean,NaN,5.0,2565.0
std,NaN,0.0,0.0
min,NaN,5.0,2565.0
25%,NaN,5.0,2565.0
50%,NaN,5.0,2565.0
75%,NaN,5.0,2565.0


In [6]:
strati_files = sorted(
    p for p in ROOT.rglob("*.xlsx")
    if p.name == "StratiTemplate.xlsx" or "Strati" in p.name or p.name.startswith(("CRidge_", "Fidelity_")) or p.name == "20250303_Hermit.xlsx"
)

strati_rows = []
for path in strati_files:
    xl = pd.ExcelFile(path)
    avy = xl.parse("AVY profile")
    tests = xl.parse("Stability Tests")
    density = xl.parse("Density")
    iris = xl.parse("IRIS")
    strati_rows.append(
        {
            "path": str(path.relative_to(ROOT)),
            "location": avy.iloc[2, 1] if avy.shape[0] > 2 and avy.shape[1] > 1 else None,
            "date": avy.iloc[0, 1] if avy.shape[0] > 0 and avy.shape[1] > 1 else None,
            "avy_shape": avy.shape,
            "tests_nonempty": int(tests.dropna(how="all").shape[0]),
            "density_rows": density.shape[0],
            "iris_rows": iris.shape[0],
        }
    )
pd.DataFrame(strati_rows)


,path,location,date,avy_shape,tests_nonempty,density_rows,iris_rows
0,Jour 1 - Fidelity\Strati_20250301_fidelity.xlsx,Fidelity full profile,20250301.0,"(46, 16)",4,61,19
1,Jour 2 - Jim Bay\20250302_JimBay_StratiTemplat...,Jim Bay Corner,20250302.0,"(29, 16)",2,61,11
2,Jour 3 - Hermit\20250303_Hermit.xlsx,Hermit wx station,20250303.0,"(47, 16)",4,61,9
3,Jour 4 - Fidelity\20250304_StratiTemplate.xlsx,Fidelity study plot,20250304.0,"(43, 16)",3,61,19
4,Jour 5 - Round Hill\20250305_Strati.xlsx,Round Hill spatial survey,20250305.0,"(55, 16)",0,61,24
5,Jour 6 - RoundHill and Christiana Ridge\CRidge...,Christiania Ridge,20250306.0,"(33, 16)",4,61,9
6,Jour 6 - RoundHill and Christiana Ridge\Fideli...,FIDELITY,202500306.0,"(52, 16)",0,61,9
7,StratiTemplate.xlsx,NaN,NaN,"(7, 16)",0,61,9


In [7]:
pnt_files = sorted(ROOT.rglob("*.pnt"))
pnt_rows = []
for path in pnt_files:
    profile = Profile.load(str(path))
    pnt_rows.append(
        {
            "path": str(path.relative_to(ROOT)),
            "samples": len(profile.samples.force),
            "max_distance_mm": float(profile.samples.distance.max()),
            "max_force_n": float(profile.samples.force.max()),
        }
    )
pd.DataFrame(pnt_rows).sort_values("samples").head(10)


Latitude value -99999.0 invalid, replacing by None (file S35M0129)


Longitude value None invalid, replacing by None (file S35M0129)


Latitude value -99999.0 invalid, replacing by None (file S35M0131)


Longitude value None invalid, replacing by None (file S35M0131)


Latitude value -99999.0 invalid, replacing by None (file S35M0133)


...

Latitude value -99999.0 invalid, replacing by None (file S35M0196)


Longitude value None invalid, replacing by None (file S35M0196)


Latitude value -99999.0 invalid, replacing by None (file S35M0205)


Longitude value None invalid, replacing by None (file S35M0205)


,path,samples,max_distance_mm,max_force_n
66,Jour 5 - Round Hill\Spatial Survey\SMP\S35M019...,3,0.008264,0.026928
68,Jour 6 - RoundHill and Christiana Ridge\SMP\S3...,16423,67.859501,1.148932
61,Jour 5 - Round Hill\Spatial Survey\SMP\S35M019...,72600,299.995855,1.128415
73,Jour 6 - RoundHill and Christiana Ridge\SMP\S3...,363000,1499.995805,8.855497
72,Jour 6 - RoundHill and Christiana Ridge\SMP\S3...,363000,1499.995805,9.574862
74,Jour 6 - RoundHill and Christiana Ridge\SMP\S3...,363000,1499.995805,8.738809
77,Jour 6 - RoundHill and Christiana Ridge\SMP\S3...,382360,1579.995802,12.421546
78,Jour 6 - RoundHill and Christiana Ridge\SMP\S3...,382360,1579.995802,9.485102
52,Jour 5 - Round Hill\Spatial Survey\SMP\S35M018...,397817,1643.867700,41.916788
8,Jour 2 - Jim Bay\Spatial Survey - SMP and Rada...,411400,1699.995797,10.446819
